In [1]:
from __future__ import annotations

from collections import Counter
import numpy as np
import networkx as nx
import mesa
from mesa.datacollection import DataCollector


class PolicyAgent(mesa.Agent):
    """
    Agente que elige una política pública bajo racionalidad limitada.

    Heurísticas implementadas:
    1) satisficing
    2) imitación del vecino más exitoso
    """

    def __init__(
        self,
        model: "PolicyDecisionModel",
        node_id: int,
        alpha: float,
        aspiration: float,
        initial_policy: int,
    ):
        super().__init__(model)
        self.node_id = node_id
        self.alpha = float(alpha)               # peso de la calidad de la política
        self.aspiration = float(aspiration)     # umbral de aspiración
        self.policy = int(initial_policy)       # política actual, valores 1..K
        self.utility = 0.0                      # utilidad realizada

        # Variables auxiliares para actualización simultánea
        self.next_policy = self.policy
        self.next_aspiration = self.aspiration

    # -------------------------------------------------------------------------
    # Información local
    # -------------------------------------------------------------------------
    def neighbors(self) -> list["PolicyAgent"]:
        """Devuelve la lista de vecinos del agente en la red."""
        return [self.model.node_to_agent[j] for j in self.model.graph.neighbors(self.node_id)]

    def local_support(self, policy: int) -> float:
        """
        Proporción de vecinos que apoyan actualmente la política `policy`.
        """
        neigh = self.neighbors()
        if not neigh:
            return 0.0
        count = sum(1 for agent in neigh if agent.policy == policy)
        return count / len(neigh)

    # -------------------------------------------------------------------------
    # Utilidad
    # -------------------------------------------------------------------------
    def evaluate_policy(self, policy: int) -> float:
        r"""
        \hat{u}_i(p,t) = \alpha_i q_p + \beta s_{ip}(t) - \gamma c_p
        """
        q_p = self.model.policy_quality[policy]
        c_p = self.model.policy_cost[policy]
        s_ip = self.local_support(policy)

        return (self.alpha * q_p) + (self.model.beta * s_ip) - (self.model.gamma * c_p)

    def update_utility(self) -> None:
        """Actualiza la utilidad realizada con la política vigente."""
        self.utility = self.evaluate_policy(self.policy)

    # -------------------------------------------------------------------------
    # Heurística de búsqueda satisficing
    # -------------------------------------------------------------------------
    def candidate_order(self) -> list[int]:
        """
        Orden secuencial de exploración de políticas.
        Se aleatoriza para no imponer siempre el mismo orden.
        """
        policies = self.model.policies.copy()
        self.random.shuffle(policies)

        if self.model.max_policy_checks is None:
            return policies
        return policies[: self.model.max_policy_checks]

    def first_satisficing_policy(self) -> tuple[int | None, float | None]:
        """
        Devuelve la primera política que satisface el umbral de aspiración actual.
        """
        for p in self.candidate_order():
            u_p = self.evaluate_policy(p)
            if u_p >= self.aspiration:
                return p, u_p
        return None, None

    # -------------------------------------------------------------------------
    # Heurística de imitación
    # -------------------------------------------------------------------------
    def best_neighbor(self) -> "PolicyAgent | None":
        """
        Selecciona al vecino con mayor utilidad observada.
        En caso de empate, elige aleatoriamente entre los mejores.
        """
        neigh = self.neighbors()
        if not neigh:
            return None

        best_u = max(agent.utility for agent in neigh)
        best_agents = [agent for agent in neigh if agent.utility == best_u]
        return self.random.choice(best_agents)

    # -------------------------------------------------------------------------
    # Dinámica temporal: actualización simultánea
    # -------------------------------------------------------------------------
    def compute_next_state(self) -> None:
        """
        Regla híbrida:
        1) si la política actual satisface el umbral, permanecer;
        2) si no satisface, buscar una política satisfactoria;
        3) si no encuentra ninguna, imitar al vecino más exitoso.
        """
        current_u = self.evaluate_policy(self.policy)
        current_a = self.aspiration

        chosen_policy = self.policy

        # Permanecer si la política actual satisface el umbral
        if current_u < current_a:
            # Búsqueda secuencial satisficing
            p_sat, _ = self.first_satisficing_policy()

            if p_sat is not None:
                chosen_policy = p_sat
            else:
                # Imitación del vecino más exitoso
                best = self.best_neighbor()
                if best is not None and best.utility > current_u:
                    chosen_policy = best.policy
                else:
                    chosen_policy = self.policy

        self.next_policy = chosen_policy

        # Actualización del umbral de aspiración:
        # a_i(t+1) = (1-rho) a_i(t) + rho * u_i(t)
        self.next_aspiration = ((1.0 - self.model.rho) * current_a) + (self.model.rho * current_u)

    def advance(self) -> None:
        """
        Segunda fase de la actualización simultánea.
        """
        self.policy = self.next_policy
        self.aspiration = self.next_aspiration


class PolicyDecisionModel(mesa.Model):
    """
    Modelo ABM de decisión sobre políticas públicas con racionalidad limitada.
    """

    def __init__(
        self,
        n_agents: int = 100,
        num_policies: int = 10,
        avg_degree: int = 6,
        rewire_prob: float = 0.10,
        beta: float = 0.80,
        gamma: float = 0.40,
        rho: float = 0.20,
        alpha_mean: float = 1.00,
        alpha_sd: float = 0.10,
        aspiration_mean: float = 0.55,
        aspiration_sd: float = 0.08,
        max_policy_checks: int | None = None,
        policy_quality: list[float] | np.ndarray | None = None,
        policy_cost: list[float] | np.ndarray | None = None,
        rng=None,
    ):
        """
        Parámetros principales
        ----------------------
        n_agents : número de agentes
        num_policies : número de políticas
        avg_degree : grado medio en la red Watts-Strogatz (debe ser par)
        rewire_prob : probabilidad de reconexión
        beta : peso del apoyo social local
        gamma : peso del costo de implementación
        rho : velocidad de ajuste del umbral de aspiración
        max_policy_checks : límite a la búsqueda secuencial (None = revisa todas)
        rng : semilla o generador aleatorio para reproducibilidad
        """
        super().__init__(rng=rng)

        # Generador robusto de NumPy, independiente de posibles inconsistencias del entorno
        self.nprng = np.random.default_rng(rng)

        if avg_degree % 2 != 0:
            raise ValueError("En Watts-Strogatz, 'avg_degree' debe ser un entero par.")
        if num_policies < 2:
            raise ValueError("'num_policies' debe ser al menos 2.")

        self.n_agents = n_agents
        self.num_policies = num_policies
        self.policies = list(range(1, num_policies + 1))
        self.beta = float(beta)
        self.gamma = float(gamma)
        self.rho = float(rho)
        self.max_policy_checks = max_policy_checks

        # ---------------------------------------------------------------------
        # Atributos de las políticas
        # ---------------------------------------------------------------------
        if policy_quality is None:
            q = self.nprng.uniform(0.40, 0.95, size=num_policies)
        else:
            q = np.asarray(policy_quality, dtype=float)

        if policy_cost is None:
            c = self.nprng.uniform(0.10, 0.70, size=num_policies)
        else:
            c = np.asarray(policy_cost, dtype=float)

        if len(q) != num_policies or len(c) != num_policies:
            raise ValueError("policy_quality y policy_cost deben tener longitud 'num_policies'.")

        self.policy_quality = {p: float(q[p - 1]) for p in self.policies}
        self.policy_cost = {p: float(c[p - 1]) for p in self.policies}

        # ---------------------------------------------------------------------
        # Red de interacción
        # ---------------------------------------------------------------------
        graph_seed = int(self.nprng.integers(0, 2**32 - 1))
        self.graph = nx.connected_watts_strogatz_graph(
            n=n_agents,
            k=avg_degree,
            p=rewire_prob,
            tries=200,
            seed=graph_seed,
        )

        # ---------------------------------------------------------------------
        # Creación de agentes
        # ---------------------------------------------------------------------
        self.node_to_agent: dict[int, PolicyAgent] = {}

        alpha_values = np.clip(
            self.nprng.normal(alpha_mean, alpha_sd, size=n_agents),
            0.10,
            None,
        )

        aspiration_values = np.clip(
            self.nprng.normal(aspiration_mean, aspiration_sd, size=n_agents),
            0.0,
            1.5,
        )

        initial_policies = self.nprng.integers(1, num_policies + 1, size=n_agents)

        for node_id, alpha_i, aspiration_i, policy_i in zip(
            self.graph.nodes(),
            alpha_values,
            aspiration_values,
            initial_policies,
        ):
            agent = PolicyAgent(
                model=self,
                node_id=node_id,
                alpha=float(alpha_i),
                aspiration=float(aspiration_i),
                initial_policy=int(policy_i),
            )
            self.node_to_agent[node_id] = agent

        # Utilidad inicial
        self.agents.do("update_utility")

        # ---------------------------------------------------------------------
        # Recolección de datos
        # ---------------------------------------------------------------------
        self.datacollector = DataCollector(
            model_reporters={
                "ImplementedPolicy": lambda m: m.implemented_policy(),
                "AverageUtility": lambda m: m.average_utility(),
                "AverageAspiration": lambda m: m.average_aspiration(),
                "DissatisfactionRate": lambda m: m.dissatisfaction_rate(),
                "ConsensusLevel": lambda m: m.consensus_level(),
                "EffectiveNumberPolicies": lambda m: m.effective_number_of_policies(),
            },
            agent_reporters={
                "Node": "node_id",
                "Policy": "policy",
                "Utility": "utility",
                "Aspiration": "aspiration",
                "Alpha": "alpha",
            },
        )

        # Registrar estado inicial
        self.datacollector.collect(self)

    # -------------------------------------------------------------------------
    # Métricas agregadas
    # -------------------------------------------------------------------------
    def policy_counts(self) -> Counter:
        return Counter(agent.policy for agent in self.agents)

    def implemented_policy(self) -> int:
        """
        Política implementada colectivamente = la más apoyada.
        Si hay empate, se elige la empatada con mayor utilidad media.
        """
        counts = self.policy_counts()
        max_count = max(counts.values())
        tied = [p for p, cnt in counts.items() if cnt == max_count]

        if len(tied) == 1:
            return tied[0]

        avg_u = {}
        for p in tied:
            vals = [agent.utility for agent in self.agents if agent.policy == p]
            avg_u[p] = float(np.mean(vals)) if vals else -np.inf

        return sorted(tied, key=lambda p: (-avg_u[p], p))[0]

    def average_utility(self) -> float:
        return float(np.mean([agent.utility for agent in self.agents]))

    def average_aspiration(self) -> float:
        return float(np.mean([agent.aspiration for agent in self.agents]))

    def dissatisfaction_rate(self) -> float:
        unsatisfied = [agent.utility < agent.aspiration for agent in self.agents]
        return float(np.mean(unsatisfied))

    def consensus_level(self) -> float:
        counts = self.policy_counts()
        return max(counts.values()) / self.n_agents

    def effective_number_of_policies(self) -> float:
        """
        Número efectivo de políticas activas:
        N_eff = 1 / sum(p_k^2)
        """
        counts = self.policy_counts()
        shares = np.array(
            [counts.get(p, 0) / self.n_agents for p in self.policies],
            dtype=float
        )
        denom = np.sum(shares ** 2)
        return float(1.0 / denom) if denom > 0 else 0.0

    # -------------------------------------------------------------------------
    # Dinámica temporal
    # -------------------------------------------------------------------------
    def step(self) -> None:
        """
        Actualización simultánea:
        1) todos calculan su siguiente estado
        2) todos avanzan
        3) todos recalculan su utilidad en el nuevo estado
        4) se recolectan datos
        """
        self.agents.do("compute_next_state")
        self.agents.do("advance")
        self.agents.do("update_utility")
        self.datacollector.collect(self)


# -----------------------------------------------------------------------------
# Ejecución de ejemplo
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    model = PolicyDecisionModel(
        n_agents=100,
        num_policies=10,
        avg_degree=6,
        rewire_prob=0.10,
        beta=0.80,
        gamma=0.40,
        rho=0.20,
        alpha_mean=1.00,
        alpha_sd=0.10,
        aspiration_mean=0.55,
        aspiration_sd=0.08,
        max_policy_checks=5,   # búsqueda acotada: revisa a lo más 5 políticas
        rng=42,
    )

    # Corrida del modelo
    n_steps = 50
    for _ in range(n_steps):
        model.step()

    # Datos del modelo
    df_model = model.datacollector.get_model_vars_dataframe()
    print("\n=== Datos agregados del modelo ===")
    print(df_model.tail())

    # Datos de agentes
    df_agents = model.datacollector.get_agent_vars_dataframe()
    print("\n=== Datos de agentes (últimas filas) ===")
    print(df_agents.tail())

    # Reconstrucción de conteos por política y por paso
    counts_by_step = (
        df_agents.reset_index()
        .groupby(["Step", "Policy"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=model.policies, fill_value=0)
    )

    print("\n=== Conteo de agentes por política (últimos pasos) ===")
    print(counts_by_step.tail())

    # Diagnóstico mínimo del estado final
    print("\n=== Diagnóstico final ===")
    print("Política implementada final:", model.implemented_policy())
    print("Utilidad promedio final:", round(model.average_utility(), 4))
    print("Aspiración promedio final:", round(model.average_aspiration(), 4))
    print("Tasa de insatisfacción final:", round(model.dissatisfaction_rate(), 4))
    print("Nivel de consenso final:", round(model.consensus_level(), 4))
    print("Número efectivo de políticas:", round(model.effective_number_of_policies(), 4))


=== Datos agregados del modelo ===
    ImplementedPolicy  AverageUtility  AverageAspiration  DissatisfactionRate  \
46                  6        0.834018           0.834006                  0.0   
47                  6        0.834018           0.834008                  0.0   
48                  6        0.834018           0.834010                  0.0   
49                  6        0.834018           0.834012                  0.0   
50                  6        0.834018           0.834013                  0.0   

    ConsensusLevel  EffectiveNumberPolicies  
46            0.22                 6.345178  
47            0.22                 6.345178  
48            0.22                 6.345178  
49            0.22                 6.345178  
50            0.22                 6.345178  

=== Datos de agentes (últimas filas) ===
              Node  Policy   Utility  Aspiration     Alpha
Step AgentID                                              
50.0 96         95       6  0.997181    0